# Nonconvex star-shaped polyhedral RAD

Original L-prism/cuboid meshes preserve every polygonal macroface and its P1 trace. The P4 local campaign uses the analytical three-dimensional RAD problem from [Araya et al. (2024)](https://doi.org/10.1016/j.cma.2024.117089), not historical connectivity. Geometry certificates, physical errors and independent quadrature controls are distinct evidence.

The current same-physics coarse control executes from source. The full P4/P1 refinement study and its galleries run with `--study`; inspect `--study --plan` for its complete locked recipe.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/2c1d9c33f8595feb2100ba72dcbb61bbb3e8085c8d10a91da8583300b18c6a8b/67_star_polyhedra-companion.zip"
COMPANION_SHA256 = "2c1d9c33f8595feb2100ba72dcbb61bbb3e8085c8d10a91da8583300b18c6a8b"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("transport/67_star_polyhedra.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
from IPython.display import Image, Markdown, display



## Current nonconvex physical case from declared equations

The n=1 L-prism/cuboid partition uses the campaign's unit-cube domain and
analytical PDE. This declared coarse control keeps local P4, original polygonal
P1, refinement 1 and assembly order 20. The full five-level campaign uses
n=2,...,6 and remains a separate refinement study. The provider exposes
coefficient integration and orientation; geometry has an explicit kernel
certificate. Original A/B/C/D rows are assembled separately from the condensed
problem and checked in the same approximation spaces as this control. No
accuracy or published-mesh agreement is inferred from this coarse execution.


In [ ]:
from inspect import getsource
from pymhm import assemble
from threadpoolctl import threadpool_limits
from examples.solve_star_polyhedra import partition
from examples.solve_rad3d import source, exact, gradient, physical_flux
from examples.formulations.transport_3d import define_transport_3d
from examples.formulations.scalar_3d import scalar_equations_3d, recover_transport_3d
from examples.formulations.original import solve_original

print(getsource(scalar_equations_3d))
mesh = partition(1)
inputs = dict(degree=4, local_refinement=1, diffusion=.1,
              velocity=(1., 0., 0.), source=source, quadrature_order=20)
with threadpool_limits(limits=1):
    definition = define_transport_3d(mesh, coarse_space="kernel", **inputs)
    system = assemble(definition.problem)
    coefficients = system.solve()
    physical = recover_transport_3d(definition, system, coefficients)
    original = solve_original(system)
    retained = define_transport_3d(mesh, coarse_space="constants", **inputs)
    retained_coefficients = assemble(retained.problem).solve()
    for actual, direct, constant in zip(coefficients.fields, original.fields,
                                        retained_coefficients.fields, strict=True):
        np.testing.assert_allclose(actual, direct, atol=1e-12, rtol=1e-10)
        np.testing.assert_allclose(actual, constant, atol=1e-12, rtol=1e-10)
    assert np.all(mesh.kernel_radii > 0)
    assert np.count_nonzero(~mesh.convex_cells) == len(mesh.cells)//2
    print({"macro_cells": len(mesh.cells), "original_faces": len(mesh.faces),
           "scalar_L2": physical.l2_error(exact, order=20),
           "gradient_L2": physical.h1_seminorm_error(gradient, order=20),
           "physical_flux_L2": physical.flux_l2_error(physical_flux, order=20)})


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    folder = ROOT / "examples/results/star-polyhedra"
    record = json.loads((folder / "comparison.json").read_text())
    assert len(record["rows"]) == 5
    assert not record["source_changed_during_run"]
    for row in record["rows"]:
        path = folder / row["archive"]
        assert hashlib.sha256(path.read_bytes()).hexdigest() == row["archive_sha256"]
        assert row["nonconvex_cells"] == row["macro_cells"] // 2
        assert row["volume_relative_difference"] < 5e-13
        assert row["kernel_radius_over_diameter_min"] > .14
        assert row["error_quadrature_change"] < 2e-9 * max(1, row["V_error"])
        with np.load(path) as fields:
            assert np.all(fields["kernel_radii"] > 0)
    header = "| n | Macrocells | Scalar L2 | V norm | Physical flux L2 |\n|---:|---:|---:|---:|---:|\n"
    table = "\n".join(f"| {r['n']} | {r['macro_cells']} | {r['l2_error']:.7g} | {r['V_error']:.7g} | {r['physical_flux_l2_error']:.7g} |" for r in record["rows"])
    display(Markdown(header + table))


The notch is part of the physical domain boundary. The marked center certifies a positive kernel ball.

In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    display(Image(filename=str(ROOT / "docs/figures/star-polyhedra/geometry.png")))

All five resolutions are retained; the geometric ratio remains bounded below in this family.

In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    display(Image(filename=str(ROOT / "docs/figures/star-polyhedra/convergence.png")))

Full local polynomials on z=0.37. Independent fine-cell values preserve jumps. Actual macro boundaries are shown on all panels.

In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    display(Image(filename=str(ROOT / "docs/figures/star-polyhedra/fields.png")))